# 01 — MySQL Telemetry SQL Walkthrough

Purpose:
This is the simple starting notebook for the MySQL telemetry lab.

It introduces:
- connecting to MySQL
- running simple SELECT queries
- inspecting tables
- previewing telemetry data
- filtering rows
- ordering rows
- reading simple JSON tags
- doing one basic JOIN to make service IDs readable

This notebook is intentionally simple.
Deeper topics can follow in later notebooks.


## Cell 2 — Install/import dependencies


In [1]:
import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

print("Imports loaded.")
print("If needed: pip install pymysql")


Imports loaded.
If needed: pip install pymysql


## Cell 3 — Connection settings


In [2]:
DB_HOST = "host.docker.internal"
DB_PORT = 3307
DB_NAME = "studybook"
DB_USER = "studybook"
DB_PASSWORD = "studybook"

DB_PORT = 3307
DB_NAME = "studybook"
DB_USER = "studybook"
DB_PASSWORD = "studybook"

password_encoded = quote_plus(DB_PASSWORD)

DATABASE_URL = (
    f"mysql+pymysql://{DB_USER}:{password_encoded}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)

print("MySQL database URL created.")

# If running inside Docker with host networking differences,
# adjust DB_HOST accordingly (for example host.docker.internal).


MySQL database URL created.


## Cell 4 — Smoke test connection


In [3]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT DATABASE(), CURRENT_USER(), NOW();"))
    row = result.fetchone()

row

('studybook', 'studybook@%', datetime.datetime(2026, 5, 11, 5, 14, 40))

## Cell 5 — Helper function to run SQL


In [4]:
def run_sql(sql: str) -> pd.DataFrame:
    """
    Run SQL against the local MySQL telemetry lab
    and return the result as a pandas DataFrame.
    """
    with engine.connect() as conn:
        return pd.read_sql_query(text(sql), conn)


## Cell 6 — Helper function to inspect one table safely


In [5]:
def inspect_table_safe(table_name: str) -> None:
    """
    Safely inspect a table without changing data.
    Shows column metadata, row count, and a small preview.
    """
    metadata_sql = f"""
    SELECT
        TABLE_SCHEMA AS table_schema,
        TABLE_NAME AS table_name,
        ORDINAL_POSITION AS ordinal_position,
        COLUMN_NAME AS column_name,
        DATA_TYPE AS data_type,
        IS_NULLABLE AS is_nullable,
        COLUMN_DEFAULT AS column_default
    FROM information_schema.columns
    WHERE TABLE_SCHEMA = DATABASE()
      AND TABLE_NAME = '{table_name}'
    ORDER BY ORDINAL_POSITION;
    """

    count_sql = f"""
    SELECT COUNT(*) AS row_count
    FROM {table_name};
    """

    preview_sql = f"""
    SELECT *
    FROM {table_name}
    LIMIT 10;
    """

    print(f"Column metadata for {table_name}")
    display(run_sql(metadata_sql))

    print(f"Row count for {table_name}")
    display(run_sql(count_sql))

    print(f"Preview rows from {table_name}")
    display(run_sql(preview_sql))


# 010 HackerRank SQL - Occupations / Pivot Occupation Column

In [6]:
# HackerRank SQL - Occupations / Pivot Occupation Column
# Purpose:
#   Drop, recreate, and populate OCCUPATIONS table for repeatable local MySQL practice.
#   Safe for repeated Jupyter runs.
#
# Supports HackerRank problems:
#   1) The PADS
#   2) Occupations Pivot

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # Doctors
    ("Samantha", "Doctor"),
    ("Jenny", "Doctor"),
    ("Amina", "Doctor"),
    ("Noah", "Doctor"),
    ("Olivia", "Doctor"),
    ("Ethan", "Doctor"),
    ("Maya", "Doctor"),
    ("Daniel", "Doctor"),

    # Professors
    ("Ashely", "Professor"),
    ("Ketty", "Professor"),
    ("Christeen", "Professor"),
    ("Robert", "Professor"),
    ("Linda", "Professor"),
    ("Farah", "Professor"),
    ("George", "Professor"),
    ("Hannah", "Professor"),
    ("Youssef", "Professor"),

    # Singers
    ("Meera", "Singer"),
    ("Priya", "Singer"),
    ("Adele", "Singer"),
    ("Bruno", "Singer"),
    ("Celine", "Singer"),
    ("Diana", "Singer"),
    ("Elena", "Singer"),

    # Actors
    ("Julia", "Actor"),
    ("Maria", "Actor"),
    ("Jane", "Actor"),
    ("Adam", "Actor"),
    ("Brian", "Actor"),
    ("Clara", "Actor"),
    ("Nora", "Actor"),
    ("Victor", "Actor"),
    ("Zain", "Actor"),
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS OCCUPATIONS;"))

    conn.execute(text("""
        CREATE TABLE OCCUPATIONS (
            Name VARCHAR(100) NOT NULL,
            Occupation VARCHAR(50) NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO OCCUPATIONS (Name, Occupation)
            VALUES (:name, :occupation);
        """),
        [{"name": name, "occupation": occupation} for name, occupation in sample_rows]
    )

print("OCCUPATIONS table dropped, recreated, and populated.")

inspect_table_safe("OCCUPATIONS")



OCCUPATIONS table dropped, recreated, and populated.
Column metadata for OCCUPATIONS


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,OCCUPATIONS,1,Name,varchar,NO,None
1,studybook,OCCUPATIONS,2,Occupation,varchar,NO,None


Row count for OCCUPATIONS


,row_count
0,33


Preview rows from OCCUPATIONS


,Name,Occupation
0,Samantha,Doctor
1,Jenny,Doctor
2,Amina,Doctor
3,Noah,Doctor
4,Olivia,Doctor
5,Ethan,Doctor
6,Maya,Doctor
7,Daniel,Doctor
8,Ashely,Professor
9,Ketty,Professor


In [11]:
sql = """
WITH Doc AS (
    SELECT ROW_NUMBER() OVER (ORDER BY Name) AS rn, Name
    FROM OCCUPATIONS
    WHERE Occupation = 'Doctor'
),
Prof AS (
    SELECT ROW_NUMBER() OVER (ORDER BY Name) AS rn, Name
    FROM OCCUPATIONS
    WHERE Occupation = 'Professor'
),
Singer AS (
    SELECT ROW_NUMBER() OVER (ORDER BY Name) AS rn, Name
    FROM OCCUPATIONS
    WHERE Occupation = 'Singer'
),
Actor AS (
    SELECT ROW_NUMBER() OVER (ORDER BY Name) AS rn, Name
    FROM OCCUPATIONS
    WHERE Occupation = 'Actor'
),
AllRows AS (
    SELECT rn FROM Doc
    UNION
    SELECT rn FROM Prof
    UNION
    SELECT rn FROM Singer
    UNION
    SELECT rn FROM Actor
)
SELECT
    Doc.Name AS Doctor,
    Prof.Name AS Professor,
    Singer.Name AS Singer,
    Actor.Name AS Actor
FROM AllRows
LEFT JOIN Doc    ON AllRows.rn = Doc.rn
LEFT JOIN Prof   ON AllRows.rn = Prof.rn
LEFT JOIN Singer ON AllRows.rn = Singer.rn
LEFT JOIN Actor  ON AllRows.rn = Actor.rn
ORDER BY AllRows.rn;
"""
display(run_sql(sql))

,Doctor,Professor,Singer,Actor
0,Amina,Ashely,Adele,Adam
1,Daniel,Christeen,Bruno,Brian
2,Ethan,Farah,Celine,Clara
3,Jenny,George,Diana,Jane
4,Maya,Hannah,Elena,Julia
5,Noah,Ketty,Meera,Maria
6,Olivia,Linda,Priya,Nora
7,Samantha,Robert,None,Victor
8,None,Youssef,None,Zain


# 011 HackerRank SQL - Binary Tree Nodes

In [12]:
# HackerRank SQL - Binary Tree Nodes
# Purpose:
#   Drop, recreate, and populate BST table for repeatable local MySQL practice.
#
# Problem:
#   Given table BST(N, P), classify each node as:
#     Root  - node has no parent, P IS NULL
#     Leaf  - node is not a parent of any other node
#     Inner - node is neither Root nor Leaf

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # N, P
    (1, 2),
    (3, 2),
    (6, 8),
    (9, 8),
    (2, 5),
    (8, 5),
    (5, None),
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS BST;"))

    conn.execute(text("""
        CREATE TABLE BST (
            N INT NOT NULL,
            P INT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO BST (N, P)
            VALUES (:n, :p);
        """),
        [{"n": n, "p": p} for n, p in sample_rows]
    )

print("BST table dropped, recreated, and populated.")

inspect_table_safe("BST")

BST table dropped, recreated, and populated.
Column metadata for BST


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,BST,1,N,int,NO,None
1,studybook,BST,2,P,int,YES,None


Row count for BST


,row_count
0,7


Preview rows from BST


,N,P
0,1,2.0
1,3,2.0
2,6,8.0
3,9,8.0
4,2,5.0
5,8,5.0
6,5,NaN


In [13]:
sql = """
SELECT 
    N,
    CASE 
        WHEN P IS NULL                        THEN 'Root'
        WHEN N NOT IN (SELECT P FROM BST 
                       WHERE P IS NOT NULL)   THEN 'Leaf'
        ELSE                                       'Inner'
    END AS NodeType
FROM BST
ORDER BY N;


"""
display(run_sql(sql))

,N,NodeType
0,1,Leaf
1,2,Inner
2,3,Leaf
3,5,Root
4,6,Leaf
5,8,Inner
6,9,Leaf


# 012 HackerRank SQL - New Companies

In [14]:
# HackerRank SQL - New Companies
# Purpose:
#   Drop, recreate, and populate company hierarchy tables for repeatable local MySQL practice.
#
# Problem:
#   For each company, print:
#     company_code
#     founder
#     count of distinct lead managers
#     count of distinct senior managers
#     count of distinct managers
#     count of distinct employees
#
# Tables:
#   Company(company_code, founder)
#   Lead_Manager(lead_manager_code, company_code)
#   Senior_Manager(senior_manager_code, lead_manager_code, company_code)
#   Manager(manager_code, senior_manager_code, lead_manager_code, company_code)
#   Employee(employee_code, manager_code, senior_manager_code, lead_manager_code, company_code)

from sqlalchemy import text
import pandas as pd

company_rows = [
    ("C1", "Monika"),
    ("C2", "Samantha"),
    ("C3", "Joseph"),
]

lead_manager_rows = [
    ("LM1", "C1"),
    ("LM2", "C2"),
    ("LM3", "C3"),
    ("LM4", "C3"),
]

senior_manager_rows = [
    ("SM1", "LM1", "C1"),
    ("SM2", "LM1", "C1"),
    ("SM3", "LM2", "C2"),
    ("SM4", "LM3", "C3"),
    ("SM5", "LM4", "C3"),
]

manager_rows = [
    ("M1", "SM1", "LM1", "C1"),
    ("M2", "SM3", "LM2", "C2"),
    ("M3", "SM3", "LM2", "C2"),
    ("M4", "SM4", "LM3", "C3"),
    ("M5", "SM5", "LM4", "C3"),
    ("M6", "SM5", "LM4", "C3"),
]

employee_rows = [
    ("E1", "M1", "SM1", "LM1", "C1"),
    ("E2", "M1", "SM1", "LM1", "C1"),
    ("E3", "M2", "SM3", "LM2", "C2"),
    ("E4", "M3", "SM3", "LM2", "C2"),
    ("E5", "M4", "SM4", "LM3", "C3"),
    ("E6", "M5", "SM5", "LM4", "C3"),
    ("E7", "M6", "SM5", "LM4", "C3"),
    ("E8", "M6", "SM5", "LM4", "C3"),
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS Employee;"))
    conn.execute(text("DROP TABLE IF EXISTS Manager;"))
    conn.execute(text("DROP TABLE IF EXISTS Senior_Manager;"))
    conn.execute(text("DROP TABLE IF EXISTS Lead_Manager;"))
    conn.execute(text("DROP TABLE IF EXISTS Company;"))

    conn.execute(text("""
        CREATE TABLE Company (
            company_code VARCHAR(20) NOT NULL,
            founder VARCHAR(100) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE Lead_Manager (
            lead_manager_code VARCHAR(20) NOT NULL,
            company_code VARCHAR(20) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE Senior_Manager (
            senior_manager_code VARCHAR(20) NOT NULL,
            lead_manager_code VARCHAR(20) NOT NULL,
            company_code VARCHAR(20) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE Manager (
            manager_code VARCHAR(20) NOT NULL,
            senior_manager_code VARCHAR(20) NOT NULL,
            lead_manager_code VARCHAR(20) NOT NULL,
            company_code VARCHAR(20) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE Employee (
            employee_code VARCHAR(20) NOT NULL,
            manager_code VARCHAR(20) NOT NULL,
            senior_manager_code VARCHAR(20) NOT NULL,
            lead_manager_code VARCHAR(20) NOT NULL,
            company_code VARCHAR(20) NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO Company (company_code, founder)
            VALUES (:company_code, :founder);
        """),
        [{"company_code": c, "founder": f} for c, f in company_rows]
    )

    conn.execute(
        text("""
            INSERT INTO Lead_Manager (lead_manager_code, company_code)
            VALUES (:lead_manager_code, :company_code);
        """),
        [
            {"lead_manager_code": lm, "company_code": c}
            for lm, c in lead_manager_rows
        ]
    )

    conn.execute(
        text("""
            INSERT INTO Senior_Manager (
                senior_manager_code,
                lead_manager_code,
                company_code
            )
            VALUES (
                :senior_manager_code,
                :lead_manager_code,
                :company_code
            );
        """),
        [
            {
                "senior_manager_code": sm,
                "lead_manager_code": lm,
                "company_code": c,
            }
            for sm, lm, c in senior_manager_rows
        ]
    )

    conn.execute(
        text("""
            INSERT INTO Manager (
                manager_code,
                senior_manager_code,
                lead_manager_code,
                company_code
            )
            VALUES (
                :manager_code,
                :senior_manager_code,
                :lead_manager_code,
                :company_code
            );
        """),
        [
            {
                "manager_code": m,
                "senior_manager_code": sm,
                "lead_manager_code": lm,
                "company_code": c,
            }
            for m, sm, lm, c in manager_rows
        ]
    )

    conn.execute(
        text("""
            INSERT INTO Employee (
                employee_code,
                manager_code,
                senior_manager_code,
                lead_manager_code,
                company_code
            )
            VALUES (
                :employee_code,
                :manager_code,
                :senior_manager_code,
                :lead_manager_code,
                :company_code
            );
        """),
        [
            {
                "employee_code": e,
                "manager_code": m,
                "senior_manager_code": sm,
                "lead_manager_code": lm,
                "company_code": c,
            }
            for e, m, sm, lm, c in employee_rows
        ]
    )

print("Company hierarchy tables dropped, recreated, and populated.")

for table_name in [
    "Company",
    "Lead_Manager",
    "Senior_Manager",
    "Manager",
    "Employee",
]:
    inspect_table_safe(table_name)

Company hierarchy tables dropped, recreated, and populated.
Column metadata for Company


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,Company,1,company_code,varchar,NO,None
1,studybook,Company,2,founder,varchar,NO,None


Row count for Company


,row_count
0,3


Preview rows from Company


...

,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,Employee,1,employee_code,varchar,NO,None
1,studybook,Employee,2,manager_code,varchar,NO,None
2,studybook,Employee,3,senior_manager_code,varchar,NO,None
3,studybook,Employee,4,lead_manager_code,varchar,NO,None
4,studybook,Employee,5,company_code,varchar,NO,None


Row count for Employee


,row_count
0,8


Preview rows from Employee


,employee_code,manager_code,senior_manager_code,lead_manager_code,company_code
0,E1,M1,SM1,LM1,C1
1,E2,M1,SM1,LM1,C1
2,E3,M2,SM3,LM2,C2
3,E4,M3,SM3,LM2,C2
4,E5,M4,SM4,LM3,C3
5,E6,M5,SM5,LM4,C3
6,E7,M6,SM5,LM4,C3
7,E8,M6,SM5,LM4,C3


In [15]:
sql = """
SELECT
    c.company_code,
    c.founder,
    COUNT(DISTINCT lm.lead_manager_code)    AS total_lead_managers,
    COUNT(DISTINCT sm.senior_manager_code)  AS total_senior_managers,
    COUNT(DISTINCT m.manager_code)          AS total_managers,
    COUNT(DISTINCT e.employee_code)         AS total_employees
FROM Company c
LEFT JOIN Lead_Manager   lm ON c.company_code = lm.company_code
LEFT JOIN Senior_Manager sm ON c.company_code = sm.company_code
LEFT JOIN Manager        m  ON c.company_code = m.company_code
LEFT JOIN Employee       e  ON c.company_code = e.company_code
GROUP BY c.company_code, c.founder
ORDER BY c.company_code;
"""
display(run_sql(sql))

,company_code,founder,total_lead_managers,total_senior_managers,total_managers,total_employees
0,C1,Monika,1,2,1,2
1,C2,Samantha,1,1,2,2
2,C3,Joseph,2,2,3,4


# 013 HackerRank SQL - Type of Triangle

In [16]:
# HackerRank SQL - Type of Triangle
# Purpose:
#   Drop, recreate, and populate TRIANGLES table for repeatable local MySQL practice.
#
# Problem:
#   Given table TRIANGLES(A, B, C), classify each row as:
#     Equilateral    - all 3 sides are equal
#     Isosceles      - exactly 2 sides are equal, or at least 2 equal after valid triangle check
#     Scalene        - all 3 sides are different
#     Not A Triangle - side lengths do not satisfy triangle inequality
#
# Key rule:
#   Check "Not A Triangle" first.

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # Sample from HackerRank
    (20, 20, 23),   # Isosceles
    (20, 20, 20),   # Equilateral
    (20, 21, 22),   # Scalene
    (13, 14, 30),   # Not A Triangle

    # Extra sanity cases
    (1, 1, 3),      # Not A Triangle, even though A = B
    (3, 4, 5),      # Scalene
    (5, 5, 8),      # Isosceles
    (7, 10, 7),     # Isosceles
    (9, 6, 6),      # Isosceles
    (10, 10, 10),   # Equilateral
    (1, 2, 3),      # Not A Triangle
    (2, 3, 10),     # Not A Triangle
    (6, 8, 10),     # Scalene
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS TRIANGLES;"))

    conn.execute(text("""
        CREATE TABLE TRIANGLES (
            A INT NOT NULL,
            B INT NOT NULL,
            C INT NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO TRIANGLES (A, B, C)
            VALUES (:a, :b, :c);
        """),
        [{"a": a, "b": b, "c": c} for a, b, c in sample_rows]
    )

print("TRIANGLES table dropped, recreated, and populated.")

inspect_table_safe("TRIANGLES")

TRIANGLES table dropped, recreated, and populated.
Column metadata for TRIANGLES


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,TRIANGLES,1,A,int,NO,None
1,studybook,TRIANGLES,2,B,int,NO,None
2,studybook,TRIANGLES,3,C,int,NO,None


Row count for TRIANGLES


,row_count
0,13


Preview rows from TRIANGLES


,A,B,C
0,20,20,23
1,20,20,20
2,20,21,22
3,13,14,30
4,1,1,3
5,3,4,5
6,5,5,8
7,7,10,7
8,9,6,6
9,10,10,10


In [23]:
sql = """
SELECT 
    CASE 
        WHEN A + B <= C OR A + C <= B OR B + C <= A THEN  'Not A Triangle'
        WHEN A = B AND B = C             THEN 'Equilateral'
        WHEN A = B OR B = C OR A = C     THEN 'Isosceles' 
        WHEN A <> B AND B <> C OR A <> C THEN 'Scalene' 
    END AS TRIANGLE_TYPE
FROM TRIANGLES ;

"""
display(run_sql(sql))

,TRIANGLE_TYPE
0,Isosceles
1,Equilateral
2,Scalene
3,Not A Triangle
4,Not A Triangle
5,Scalene
6,Isosceles
7,Isosceles
8,Isosceles
9,Equilateral


# 014 HackerRank SQL - The Blunder

In [24]:
# HackerRank SQL - The Blunder
# Purpose:
#   Drop, recreate, and populate EMPLOYEES table for repeatable local MySQL practice.
#
# Problem:
#   Samantha calculated average salary incorrectly because her keyboard's 0 key was broken.
#   For each salary, remove all 0 digits, calculate the wrong average,
#   then output CEIL(actual_average - wrong_average).

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # HackerRank sample
    (1, "Kristeen", 1420),
    (2, "Ashley", 2006),
    (3, "Julia", 2210),
    (4, "Maria", 3000),

    # Extra sanity rows
    (5, "Samantha", 3050),
    (6, "Jenny", 4005),
    (7, "Priya", 10000),
    (8, "Meera", 9876),
    (9, "Jane", 7070),
    (10, "Amina", 1203),
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS EMPLOYEES;"))

    conn.execute(text("""
        CREATE TABLE EMPLOYEES (
            ID INT NOT NULL,
            Name VARCHAR(100) NOT NULL,
            Salary INT NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO EMPLOYEES (ID, Name, Salary)
            VALUES (:id, :name, :salary);
        """),
        [
            {"id": employee_id, "name": name, "salary": salary}
            for employee_id, name, salary in sample_rows
        ]
    )

print("EMPLOYEES table dropped, recreated, and populated.")

inspect_table_safe("EMPLOYEES")

EMPLOYEES table dropped, recreated, and populated.
Column metadata for EMPLOYEES


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,EMPLOYEES,1,ID,int,NO,None
1,studybook,EMPLOYEES,2,Name,varchar,NO,None
2,studybook,EMPLOYEES,3,Salary,int,NO,None


Row count for EMPLOYEES


,row_count
0,10


Preview rows from EMPLOYEES


,ID,Name,Salary
0,1,Kristeen,1420
1,2,Ashley,2006
2,3,Julia,2210
3,4,Maria,3000
4,5,Samantha,3050
5,6,Jenny,4005
6,7,Priya,10000
7,8,Meera,9876
8,9,Jane,7070
9,10,Amina,1203


In [27]:
sql = """
SELECT CEIL(
    AVG(Salary) - AVG(CAST(REPLACE(Salary, '0', '') AS UNSIGNED))
) AS difference
FROM EMPLOYEES;
"""
display(run_sql(sql))

,difference
0,3330.0


# 015 HackerRank SQL - Top Earners

In [28]:
# HackerRank SQL - Top Earners
# Purpose:
#   Drop, recreate, and populate Employee table for repeatable local MySQL practice.
#
# Problem:
#   Each employee's total earnings = months * salary.
#   Output:
#     maximum total earnings
#     count of employees with that maximum total earnings
#
# Expected output shape:
#   max_earnings employee_count

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # HackerRank sample
    (12228, "Rose", 15, 1968),
    (33645, "Angela", 1, 3443),
    (45692, "Frank", 17, 1608),
    (56118, "Patrick", 7, 1345),
    (59725, "Lisa", 11, 2330),
    (74197, "Kimberly", 16, 4372),
    (78454, "Bonnie", 8, 1771),
    (83565, "Michael", 6, 2017),
    (98607, "Todd", 5, 3396),
    (99989, "Joe", 9, 3573),

    # Extra sanity rows
    (10001, "Amina", 10, 5000),     # 50000
    (10002, "Noah", 20, 2000),      # 40000
    (10003, "Priya", 8, 6000),      # 48000
    (10004, "Daniel", 16, 4372),    # 69952, tie with Kimberly
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS Employee;"))

    conn.execute(text("""
        CREATE TABLE Employee (
            employee_id INT NOT NULL,
            name VARCHAR(100) NOT NULL,
            months INT NOT NULL,
            salary INT NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO Employee (employee_id, name, months, salary)
            VALUES (:employee_id, :name, :months, :salary);
        """),
        [
            {
                "employee_id": employee_id,
                "name": name,
                "months": months,
                "salary": salary,
            }
            for employee_id, name, months, salary in sample_rows
        ]
    )

print("Employee table dropped, recreated, and populated.")

inspect_table_safe("Employee")

Employee table dropped, recreated, and populated.
Column metadata for Employee


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,Employee,1,employee_id,int,NO,None
1,studybook,Employee,2,name,varchar,NO,None
2,studybook,Employee,3,months,int,NO,None
3,studybook,Employee,4,salary,int,NO,None


Row count for Employee


,row_count
0,14


Preview rows from Employee


,employee_id,name,months,salary
0,12228,Rose,15,1968
1,33645,Angela,1,3443
2,45692,Frank,17,1608
3,56118,Patrick,7,1345
4,59725,Lisa,11,2330
5,74197,Kimberly,16,4372
6,78454,Bonnie,8,1771
7,83565,Michael,6,2017
8,98607,Todd,5,3396
9,99989,Joe,9,3573


In [36]:
sql = """
SELECT (months * salary) AS earnings, COUNT(*)
FROM Employee
GROUP BY earnings
ORDER BY earnings DESC
LIMIT 1;
"""
display(run_sql(sql))

,earnings,COUNT(*)
0,69952,2


In [35]:
sql = """
WITH max_earn AS (
    SELECT MAX(months * salary) as max_earnings
    FROM Employee
)
SELECT max_earnings 
FROM max_earn;
"""
display(run_sql(sql))

,max_earnings
0,69952


# 017 HackerRank SQL - Weather Observation Station 18/19 style practice

In [37]:
# HackerRank SQL - Weather Observation Station 18/19 style practice
# Specific problem:
#   Query the Western Longitude (LONG_W) where the smallest Northern Latitude (LAT_N)
#   in STATION is greater than 38.7780.
#   Round the answer to 4 decimal places.
#
# Purpose:
#   Drop, recreate, and populate STATION table for repeatable local MySQL practice.

from sqlalchemy import text
import pandas as pd

sample_rows = [
    # ID, CITY, STATE, LAT_N, LONG_W
    (1,  "Dallas",      "TX", 32.7767, 96.7970),
    (2,  "Austin",      "TX", 30.2672, 97.7431),
    (3,  "Houston",     "TX", 29.7604, 95.3698),
    (4,  "Chicago",     "IL", 41.8781, 87.6298),
    (5,  "Denver",      "CO", 39.7392, 104.9903),
    (6,  "Boston",      "MA", 42.3601, 71.0589),

    # Important test rows around 38.7780
    (7,  "BelowPoint",  "AA", 38.7779, 120.1111),
    (8,  "TargetCity",  "BB", 38.7781, 117.246456),
    (9,  "HigherCity",  "CC", 38.9000, 118.9999),
    (10, "FarHigher",   "DD", 50.0000, 130.1234),
]

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS STATION;"))

    conn.execute(text("""
        CREATE TABLE STATION (
            ID INT NOT NULL,
            CITY VARCHAR(21) NOT NULL,
            STATE VARCHAR(2) NOT NULL,
            LAT_N DECIMAL(10, 6) NOT NULL,
            LONG_W DECIMAL(10, 6) NOT NULL
        );
    """))

    conn.execute(
        text("""
            INSERT INTO STATION (ID, CITY, STATE, LAT_N, LONG_W)
            VALUES (:id, :city, :state, :lat_n, :long_w);
        """),
        [
            {
                "id": row_id,
                "city": city,
                "state": state,
                "lat_n": lat_n,
                "long_w": long_w,
            }
            for row_id, city, state, lat_n, long_w in sample_rows
        ]
    )

print("STATION table dropped, recreated, and populated.")

inspect_table_safe("STATION")

STATION table dropped, recreated, and populated.
Column metadata for STATION


,table_schema,table_name,ordinal_position,column_name,data_type,is_nullable,column_default
0,studybook,STATION,1,ID,int,NO,None
1,studybook,STATION,2,CITY,varchar,NO,None
2,studybook,STATION,3,STATE,varchar,NO,None
3,studybook,STATION,4,LAT_N,decimal,NO,None
4,studybook,STATION,5,LONG_W,decimal,NO,None


Row count for STATION


,row_count
0,10


Preview rows from STATION


,ID,CITY,STATE,LAT_N,LONG_W
0,1,Dallas,TX,32.7767,96.797000
1,2,Austin,TX,30.2672,97.743100
2,3,Houston,TX,29.7604,95.369800
3,4,Chicago,IL,41.8781,87.629800
4,5,Denver,CO,39.7392,104.990300
5,6,Boston,MA,42.3601,71.058900
6,7,BelowPoint,AA,38.7779,120.111100
7,8,TargetCity,BB,38.7781,117.246456
8,9,HigherCity,CC,38.9000,118.999900
9,10,FarHigher,DD,50.0000,130.123400


In [38]:
sql = """
SELECT ROUND(LONG_W, 4)
FROM STATION
WHERE LAT_N > 38.7780
ORDER BY LAT_N ASC
LIMIT 1;
"""
display(run_sql(sql))

,"ROUND(LONG_W, 4)"
0,117.2465


In [39]:
sql = """
SET @rows = 5;

SELECT REPEAT('* ', n)
FROM (
    SELECT @rownum := @rownum + 1 AS n
    FROM information_schema.columns,
         (SELECT @rownum := 0) r
    LIMIT 5
) numbered;
"""
display(run_sql(sql))

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'SELECT REPEAT('* ', n)\nFROM (\n    SELECT @rownum := @rownum + 1 AS n\n    FROM in' at line 3")
[SQL: 
SET @rows = 5;

SELECT REPEAT('* ', n)
FROM (
    SELECT @rownum := @rownum + 1 AS n
    FROM information_schema.columns,
         (SELECT @rownum := 0) r
    LIMIT 5
) numbered;
]
(Background on this error at: https://sqlalche.me/e/20/f405)